# Polygon.io (Massive) — Data Ingestion

Pulls **2 years** of daily bars for stocks, options chain snapshots with greeks, and futures.  
Rate-limited to **5 req/s** with automatic retry (exponential back-off) and **checkpoint/resume** (existing parquet files are skipped).

```
data/raw/
  stocks/           {TICKER}_day.parquet
  options/
    snapshots/      {UNDERLYING}_{DATE}.parquet   <- greeks, IV, OI
    history/        {CONTRACT}_day.parquet        <- filtered OHLCV
  futures/          {CONTRACT}_day.parquet
```

In [ ]:
%pip install aiohttp nest-asyncio python-dotenv tqdm pandas pyarrow --quiet

In [ ]:
import os
import asyncio
import logging
import time
from datetime import date, timedelta
from pathlib import Path
from typing import Optional

import aiohttp
import nest_asyncio
import pandas as pd
from dotenv import load_dotenv
from tqdm.auto import tqdm

nest_asyncio.apply()  # allow asyncio.run() inside Jupyter

load_dotenv('../.env')

API_KEY    = os.environ['MARKET_DATA_API_KEY']
BASE_URL   = 'https://api.polygon.io'
RATE_LIMIT = 5.0   # requests per second

END_DATE   = date.today().isoformat()
START_DATE = (date.today() - timedelta(days=730)).isoformat()

DATA_DIR = Path('../data/raw')
for sub in ('stocks', 'options/snapshots', 'options/history', 'futures'):
    (DATA_DIR / sub).mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('polygon_ingest')

print(f'Date range: {START_DATE}  ->  {END_DATE}')

In [ ]:
class TokenBucket:
    """Async token-bucket rate limiter."""

    def __init__(self, rate: float):
        self.rate = rate
        self.tokens = float(rate)
        self._last = time.monotonic()
        self._lock = asyncio.Lock()

    async def acquire(self) -> None:
        async with self._lock:
            now = time.monotonic()
            self.tokens = min(self.rate, self.tokens + (now - self._last) * self.rate)
            self._last = now
            if self.tokens < 1.0:
                await asyncio.sleep((1.0 - self.tokens) / self.rate)
                self.tokens = 0.0
            else:
                self.tokens -= 1.0

In [ ]:
class PolygonClient:
    _MAX_RETRIES = 5

    def __init__(self, api_key: str, rate: float = 5.0):
        self.api_key = api_key
        self._limiter = TokenBucket(rate)
        self._session: Optional[aiohttp.ClientSession] = None

    async def _get_session(self) -> aiohttp.ClientSession:
        if self._session is None or self._session.closed:
            self._session = aiohttp.ClientSession(
                headers={'Authorization': f'Bearer {self.api_key}'},
                timeout=aiohttp.ClientTimeout(total=30),
            )
        return self._session

    async def close(self) -> None:
        if self._session and not self._session.closed:
            await self._session.close()

    async def get(self, url: str, params: Optional[dict] = None) -> dict:
        session = await self._get_session()
        for attempt in range(self._MAX_RETRIES):
            await self._limiter.acquire()
            try:
                async with session.get(url, params=params) as resp:
                    if resp.status == 429:
                        wait = 2 ** attempt
                        log.warning('429 rate-limit; sleeping %ds', wait)
                        await asyncio.sleep(wait)
                        continue
                    resp.raise_for_status()
                    return await resp.json()
            except (aiohttp.ClientError, asyncio.TimeoutError) as exc:
                if attempt == self._MAX_RETRIES - 1:
                    raise
                wait = 2 ** attempt
                log.warning('Error (attempt %d): %s -- retry in %ds', attempt + 1, exc, wait)
                await asyncio.sleep(wait)
        return {}

    async def paginate(self, url: str, params: Optional[dict] = None) -> list:
        """Follow Polygon next_url cursors to collect all pages."""
        all_results = []
        first = True
        while url:
            data = await self.get(url, params if first else None)
            first = False
            all_results.extend(data.get('results') or [])
            url = data.get('next_url', '')
        return all_results


client = PolygonClient(API_KEY, RATE_LIMIT)
print('Client ready.')

## 1 - Stocks

One paginated request per ticker — 2 years of daily bars fit in a single call (Polygon returns up to 50 000 bars per page).  
Existing `.parquet` files are skipped on re-runs.

In [ ]:
STOCK_TICKERS = [
    # Core large-cap universe -- edit to match your strategy
    'AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL', 'META', 'TSLA',
    'JPM',  'V',    'MA',   'BAC',  'GS',    'MS',
    'XOM',  'CVX',  'COP',
    'LLY',  'UNH',  'PFE',  'MRK',  'ABBV',
    'AVGO', 'AMD',  'INTC', 'QCOM', 'TXN',
    'HD',   'WMT',  'COST', 'TGT',
    'CAT',  'RTX',  'HON',  'DE',
    'NEE',  'DUK',  'SO',
    # Broad ETFs
    'SPY',  'QQQ',  'IWM',  'DIA',  'GLD',  'SLV',  'USO',  'TLT',
]
STOCK_TICKERS = sorted(set(STOCK_TICKERS))
print(f'{len(STOCK_TICKERS)} tickers in universe')

In [ ]:
async def fetch_stock_bars(
    ticker: str,
    timespan: str = 'day',
    multiplier: int = 1,
) -> pd.DataFrame:
    out = DATA_DIR / 'stocks' / f'{ticker}_{timespan}.parquet'
    if out.exists():
        return pd.read_parquet(out)

    url = (
        f'{BASE_URL}/v2/aggs/ticker/{ticker}/range'
        f'/{multiplier}/{timespan}/{START_DATE}/{END_DATE}'
    )
    rows = await client.paginate(url, {'adjusted': 'true', 'sort': 'asc'})
    if not rows:
        log.warning('No bars: %s', ticker)
        return pd.DataFrame()

    df = pd.DataFrame(rows).rename(columns={
        't': 'ts', 'o': 'open', 'h': 'high', 'l': 'low',
        'c': 'close', 'v': 'volume', 'vw': 'vwap', 'n': 'transactions',
    })
    df['ts'] = pd.to_datetime(df['ts'], unit='ms', utc=True)
    df['ticker'] = ticker
    df = df.set_index('ts')[['ticker', 'open', 'high', 'low', 'close', 'volume', 'vwap', 'transactions']]
    df.to_parquet(out)
    return df


async def ingest_stocks(timespans=None):
    timespans = timespans or ['day']
    pairs = [(t, ts) for t in STOCK_TICKERS for ts in timespans]
    results = {}
    for ticker, timespan in tqdm(pairs, desc='Stocks'):
        results[f'{ticker}_{timespan}'] = await fetch_stock_bars(ticker, timespan)
    return results

In [ ]:
stock_data = asyncio.run(ingest_stocks(['day']))

ok = {k: v for k, v in stock_data.items() if not v.empty}
print(f'\nFetched {len(ok)}/{len(stock_data)} tickers')
if ok:
    k0 = next(iter(ok))
    print(f'\nSample -- {k0}:')
    print(ok[k0].tail(3).to_string())

## 2 - Options

**Snapshot** (`/v3/snapshot/options/{underlying}`): full chain with greeks, IV, and open interest saved once per day per underlying.  
**History** (`/v2/aggs`): daily OHLCV for a *filtered* subset of contracts only — controlled by the parameters below.

> Without filters, fetching history for all AAPL contracts is ~10 000 requests.  
> The default filters (0-90 DTE, OI >= 100, strike +/- 20% of spot) are a reasonable starting point.

In [ ]:
OPTIONS_UNDERLYINGS = ['AAPL', 'MSFT', 'NVDA', 'SPY', 'QQQ', 'TSLA', 'AMZN', 'GOOGL']

OPT_MIN_DTE    = 0    # minimum days to expiry
OPT_MAX_DTE    = 90   # maximum days to expiry
OPT_MIN_OI     = 100  # minimum open interest
OPT_STRIKE_PCT = 20   # +/- % of underlying spot price

print(f'Underlyings: {OPTIONS_UNDERLYINGS}')
print(f'History filters: DTE {OPT_MIN_DTE}-{OPT_MAX_DTE}, OI >= {OPT_MIN_OI}, strike +/-{OPT_STRIKE_PCT}% spot')

In [ ]:
def _flatten_chain(rows: list, underlying: str, snapshot_date: str) -> pd.DataFrame:
    records = []
    for r in rows:
        det = r.get('details') or {}
        grk = r.get('greeks') or {}
        day = r.get('day') or {}
        lq  = r.get('last_quote') or {}
        records.append({
            'snapshot_date':       snapshot_date,
            'underlying':          underlying,
            'ticker':              det.get('ticker'),
            'contract_type':       det.get('contract_type'),
            'expiration_date':     det.get('expiration_date'),
            'strike_price':        det.get('strike_price'),
            'exercise_style':      det.get('exercise_style'),
            'shares_per_contract': det.get('shares_per_contract'),
            'open_interest':       r.get('open_interest'),
            'implied_volatility':  r.get('implied_volatility'),
            'delta': grk.get('delta'),
            'gamma': grk.get('gamma'),
            'theta': grk.get('theta'),
            'vega':  grk.get('vega'),
            'day_open':         day.get('open'),
            'day_high':         day.get('high'),
            'day_low':          day.get('low'),
            'day_close':        day.get('close'),
            'day_volume':       day.get('volume'),
            'day_vwap':         day.get('vwap'),
            'bid':              lq.get('bid'),
            'ask':              lq.get('ask'),
            'break_even_price': r.get('break_even_price'),
        })
    return pd.DataFrame(records)


async def fetch_options_chain(underlying: str) -> pd.DataFrame:
    today = date.today().isoformat()
    out = DATA_DIR / 'options' / 'snapshots' / f'{underlying}_{today}.parquet'
    if out.exists():
        return pd.read_parquet(out)

    url = f'{BASE_URL}/v3/snapshot/options/{underlying}'
    rows = await client.paginate(url, {'limit': 250})
    if not rows:
        log.warning('No chain data: %s', underlying)
        return pd.DataFrame()

    df = _flatten_chain(rows, underlying, today)
    df.to_parquet(out, index=False)
    return df


async def ingest_options_chains():
    chains = {}
    for und in tqdm(OPTIONS_UNDERLYINGS, desc='Options snapshots'):
        chains[und] = await fetch_options_chain(und)
    return chains

In [ ]:
chain_data = asyncio.run(ingest_options_chains())

ok_chains = {k: v for k, v in chain_data.items() if not v.empty}
print(f'\nChains: {len(ok_chains)}/{len(OPTIONS_UNDERLYINGS)} underlyings')
for und, df in ok_chains.items():
    print(f'  {und}: {len(df):,} contracts')

In [ ]:
def _filter_contracts(chain_df: pd.DataFrame, spot: float) -> list:
    """Return option tickers that pass DTE, OI, and moneyness filters."""
    today = pd.Timestamp.today().normalize()
    df = chain_df.copy()
    df['expiration_date'] = pd.to_datetime(df['expiration_date'])
    df['dte'] = (df['expiration_date'] - today).dt.days
    mask = (
        df['dte'].between(OPT_MIN_DTE, OPT_MAX_DTE)
        & (df['open_interest'] >= OPT_MIN_OI)
        & (df['strike_price'] >= spot * (1 - OPT_STRIKE_PCT / 100))
        & (df['strike_price'] <= spot * (1 + OPT_STRIKE_PCT / 100))
    )
    return df.loc[mask, 'ticker'].dropna().tolist()


async def _fetch_prev_close(ticker: str) -> float:
    data = await client.get(f'{BASE_URL}/v2/aggs/ticker/{ticker}/prev')
    results = data.get('results') or [{}]
    return float(results[0].get('c', 0)) if results else 0.0


async def fetch_options_bar(contract_ticker: str) -> pd.DataFrame:
    safe = contract_ticker.replace(':', '_').replace('/', '_')
    out = DATA_DIR / 'options' / 'history' / f'{safe}_day.parquet'
    if out.exists():
        return pd.read_parquet(out)

    url = (
        f'{BASE_URL}/v2/aggs/ticker/{contract_ticker}/range'
        f'/1/day/{START_DATE}/{END_DATE}'
    )
    rows = await client.paginate(url, {'adjusted': 'false', 'sort': 'asc'})
    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows).rename(columns={
        't': 'ts', 'o': 'open', 'h': 'high', 'l': 'low',
        'c': 'close', 'v': 'volume', 'vw': 'vwap', 'n': 'transactions',
    })
    df['ts'] = pd.to_datetime(df['ts'], unit='ms', utc=True)
    df['ticker'] = contract_ticker
    df = df.set_index('ts')[['ticker', 'open', 'high', 'low', 'close', 'volume', 'vwap', 'transactions']]
    df.to_parquet(out)
    return df


async def ingest_options_history(chain_data: dict) -> dict:
    spots = {und: await _fetch_prev_close(und) for und in OPTIONS_UNDERLYINGS}

    contracts = []
    for und, df in chain_data.items():
        if not df.empty and spots.get(und, 0) > 0:
            contracts.extend(_filter_contracts(df, spots[und]))
    contracts = list(set(contracts))
    log.info('Fetching history for %d filtered option contracts', len(contracts))

    history = {}
    for ticker in tqdm(contracts, desc='Options history'):
        history[ticker] = await fetch_options_bar(ticker)
    return history

In [ ]:
options_history = asyncio.run(ingest_options_history(chain_data))

ok_hist = {k: v for k, v in options_history.items() if not v.empty}
total_bars = sum(len(v) for v in ok_hist.values())
print(f'\nHistory: {len(ok_hist)}/{len(options_history)} contracts  |  {total_bars:,} total bars')

## 3 - Futures

Polygon continuous-contract tickers use the `{ROOT}1!` convention (front month).  
**Verify your exact ticker symbols** against the Polygon/Massive reference API before running — the format can differ from the examples below.
Use `/v3/reference/tickers?market=futures` or the Massive dashboard to confirm.

In [ ]:
FUTURES_TICKERS = [
    # Equity index
    'ES1!',   # E-mini S&P 500
    'NQ1!',   # E-mini Nasdaq-100
    'RTY1!',  # E-mini Russell 2000
    'YM1!',   # E-mini Dow Jones
    # Energy
    'CL1!',   # WTI Crude Oil
    'NG1!',   # Natural Gas
    # Metals
    'GC1!',   # Gold
    'SI1!',   # Silver
    'HG1!',   # Copper
    # Rates
    'ZB1!',   # 30-Year T-Bond
    'ZN1!',   # 10-Year T-Note
    # Grains
    'ZC1!',   # Corn
    'ZS1!',   # Soybeans
    'ZW1!',   # Wheat
]
print(f'{len(FUTURES_TICKERS)} futures contracts configured')

In [ ]:
async def fetch_futures_bars(
    ticker: str,
    timespan: str = 'day',
    multiplier: int = 1,
) -> pd.DataFrame:
    safe = ticker.replace('!', 'cont').replace(':', '_').replace('/', '_')
    out = DATA_DIR / 'futures' / f'{safe}_{timespan}.parquet'
    if out.exists():
        return pd.read_parquet(out)

    url = (
        f'{BASE_URL}/v2/aggs/ticker/{ticker}/range'
        f'/{multiplier}/{timespan}/{START_DATE}/{END_DATE}'
    )
    rows = await client.paginate(url, {'adjusted': 'true', 'sort': 'asc'})
    if not rows:
        log.warning('No data: %s -- verify ticker format', ticker)
        return pd.DataFrame()

    df = pd.DataFrame(rows).rename(columns={
        't': 'ts', 'o': 'open', 'h': 'high', 'l': 'low',
        'c': 'close', 'v': 'volume', 'vw': 'vwap', 'n': 'transactions',
    })
    df['ts'] = pd.to_datetime(df['ts'], unit='ms', utc=True)
    df['ticker'] = ticker
    df = df.set_index('ts')[['ticker', 'open', 'high', 'low', 'close', 'volume', 'vwap', 'transactions']]
    df.to_parquet(out)
    return df


async def ingest_futures(timespans=None):
    timespans = timespans or ['day']
    pairs = [(t, ts) for t in FUTURES_TICKERS for ts in timespans]
    results = {}
    for ticker, timespan in tqdm(pairs, desc='Futures'):
        results[f'{ticker}_{timespan}'] = await fetch_futures_bars(ticker, timespan)
    return results

In [ ]:
futures_data = asyncio.run(ingest_futures(['day']))

ok_fut = {k: v for k, v in futures_data.items() if not v.empty}
print(f'\nFetched {len(ok_fut)}/{len(futures_data)} futures')
if ok_fut:
    k0 = next(iter(ok_fut))
    print(f'\nSample -- {k0}:')
    print(ok_fut[k0].tail(3).to_string())

## 4 - Validation

In [ ]:
print('=' * 62)
print('INGESTION SUMMARY')
print('=' * 62)

stock_ok = {k: v for k, v in stock_data.items()      if not v.empty}
chain_ok = {k: v for k, v in chain_data.items()      if not v.empty}
hist_ok  = {k: v for k, v in options_history.items() if not v.empty}
fut_ok   = {k: v for k, v in futures_data.items()    if not v.empty}

print(f'\nStocks:          {len(stock_ok):>4} tickers      {sum(len(v) for v in stock_ok.values()):>10,} bars')
print(f'Options chains:  {len(chain_ok):>4} underlyings  {sum(len(v) for v in chain_ok.values()):>10,} contracts')
print(f'Options history: {len(hist_ok):>4} contracts    {sum(len(v) for v in hist_ok.values()):>10,} bars')
print(f'Futures:         {len(fut_ok):>4} contracts    {sum(len(v) for v in fut_ok.values()):>10,} bars')

print('\n--- Null % in OHLCV (spot-check first 3 per class) ---')
for label, d in [('Stocks', stock_ok), ('Futures', fut_ok)]:
    for key, df in list(d.items())[:3]:
        cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in df.columns]
        null_pct = df[cols].isnull().mean().mean() * 100
        date_min = df.index.min().date() if not df.empty else 'n/a'
        date_max = df.index.max().date() if not df.empty else 'n/a'
        print(f'  {label} {key}: {len(df)} rows  {null_pct:.1f}% nulls  [{date_min} -> {date_max}]')

print(f'\nData written to: {DATA_DIR.resolve()}')

In [ ]:
asyncio.run(client.close())
print('HTTP session closed.')